In [1]:
%config Completer.use_jedi = False # fixes the jupyter notebook issue where tab fails to auto-complete internal functions

In [1]:
import os
import shutil
import copy as cp
import pandas as pd 
import numpy as np
from IPython.display import clear_output

In [2]:
cwd0=os.getcwd()
cwd_interim=os.path.join(cwd0,'interim_files/')
cwd_refdata=os.path.join(cwd0,'reference_data_files/')


In [3]:
# Find Studies
# cwd='C:/Users/GM/Downloads/Genomics_Analysis_FREEZE4_GM_Apr2021/cbioportal_raw_EXOME142_Mar2021'
cwd_raw_data=os.path.join(cwd0,'cbioportal_raw_EXOME139') # fresh cbioportal download directory
# list_study_dirs=['acbc_mskcc_2015','acc_tcga_pan_can_atlas_2018','acyc_mda_2015','acyc_mskcc_2013','acyc_sanger_2013','all_stjude_2013','all_stjude_2015','all_stjude_2016','aml_target_2018_pub','angs_project_painter_2018','bfn_duke_nus_2015','blca_bgi','blca_cornell_2016','blca_dfarber_mskcc_2014','blca_tcga_pub_2017','brca_bccrc','brca_broad','brca_igr_2015','brca_mbcproject_wagle_2017','brca_sanger','brca_tcga_pan_can_atlas_2018','ccrcc_irc_2014','ccrcc_utokyo_2013','cesc_tcga_pan_can_atlas_2018','chol_jhu_2013','chol_nccs_2013','chol_tcga_pan_can_atlas_2018','cll_iuopa_2015','cllsll_icgc_2011','coadread_dfci_2016','coadread_genentech','coadread_tcga_pan_can_atlas_2018','ctcl_columbia_2015','desm_broad_2015','dlbc_broad_2012','dlbc_tcga_pan_can_atlas_2018','dlbcl_dfci_2018','dlbcl_duke_2017','egc_tmucih_2015','es_dfarber_broad_2014','es_iocurie_2014','esca_tcga_pan_can_atlas_2018','escc_icgc','escc_ucla_2014','gbc_shanghai_2014','gbm_tcga_pan_can_atlas_2018','hcc_inserm_fr_2015','hnsc_broad','hnsc_jhu','hnsc_mdanderson_2013','hnsc_tcga_pan_can_atlas_2018','kich_tcga_pan_can_atlas_2018','kirc_bgi','kirc_tcga_pan_can_atlas_2018','kirp_tcga_pan_can_atlas_2018','laml_tcga_pan_can_atlas_2018','lcll_broad_2013','lgg_ucsf_2014','lgggbm_tcga_pub','lihc_amc_prv','lihc_riken','lihc_tcga_pan_can_atlas_2018','luad_broad','luad_mskcc_2015','luad_tcga_pan_can_atlas_2018','lusc_tcga_pan_can_atlas_2018','mbl_broad_2012','mbl_icgc','mbl_pcgp','mbl_sickkids_2016','mcl_idibips_2013','mds_tokyo_2011','mel_tsam_liang_2017','meso_tcga_pan_can_atlas_2018','mm_broad','mpnst_mskcc','mrt_bcgsc_2016','nbl_amc_2012','nbl_broad_2013','nbl_target_2018_pub','nbl_ucologne_2015','nccrcc_genentech_2014','nepc_wcm_2016','nhl_bcgsc_2011','nhl_bcgsc_2013','npc_nusingapore','nsclc_tcga_broad_2016','ov_tcga_pan_can_atlas_2018','paac_jhu_2014','paad_icgc','paad_qcmg_uq_2016','paad_tcga_pan_can_atlas_2018','paad_utsw_2015','panet_arcnet_2017','panet_jhu_2011','panet_shanghai_2013','past_dkfz_heidelberg_2013','pcnsl_mayo_2015','pcpg_tcga_pan_can_atlas_2018','plmeso_nyu_2015','prad_broad','prad_broad_2013','prad_cpcg_2017','prad_eururol_2017','prad_fhcrc','prad_mich','prad_p1000','prad_su2c_2015','prad_tcga_pan_can_atlas_2018','rms_nih_2014','rt_target_2018_pub','sarc_tcga_pan_can_atlas_2018','sclc_cancercell_gardner_2017','sclc_clcgp','sclc_jhu','sclc_ucologne_2015','skcm_broad','skcm_broad_brafresist_2012','skcm_broad_dfarber','skcm_tcga_pan_can_atlas_2018','skcm_ucla_2016','skcm_yale','stad_pfizer_uhongkong','stad_tcga_pan_can_atlas_2018','stad_uhongkong','stad_utokyo','stes_tcga_pub','tet_nci_2014','tgct_tcga_pan_can_atlas_2018','thca_tcga_pan_can_atlas_2018','thym_tcga_pan_can_atlas_2018','uccc_nih_2017','ucec_tcga_pan_can_atlas_2018','ucs_jhu_2014','ucs_tcga_pan_can_atlas_2018','um_qimr_2016','uvm_tcga_pan_can_atlas_2018','vsc_cuk_2018','wt_target_2018_pub']
# print('Number of Studies included:',len(list_study_dirs),'\n The following studies are not present in cbioportal_raw_EXOME139/ folder. Please download by running script provided in the folder: ',[istudy for istudy in list_study_dirs if not os.path.isdir(os.path.join(cwd_raw_data,istudy))]) # Second value should be null vector to ensure all studies are downloaded and unzipped in current directory.
list_study_dirs=[idir for idir in os.listdir(cwd_raw_data) if os.path.isdir(os.path.join(cwd_raw_data,idir))]
print('Number of Studies included:',len(list_study_dirs))

Number of Studies included: 139


#### Some USeful FUnctions

In [4]:
# Define Useful Functions 
setlen=lambda x:len(set(x)) # Calculate length of set of a list.

### Part 1: Curated Clinical Data

In [5]:
%%time
# Import Clinical Data
# when clinical folder defined in cwd0 directory
cldirlist0=[os.path.join(cwd0,'clinical',idir) for idir in os.listdir(os.path.join(cwd0,"clinical")) if os.path.isdir(os.path.join(cwd0,'clinical',idir))]
cldirlist1=[os.path.join(idir,idir1) for idir in cldirlist0 for idir1 in os.listdir(idir)]

# limit to included studies
clinical_file_list=[os.path.join(idir,'data_clinical_sample_v2.txt') for idir in cldirlist1 if any([row in idir for row in list_study_dirs])] 

N_LongiSamples={}
list_df_clinical=[]
for istudypath in clinical_file_list:
    Study_ID=istudypath.split('\\')[-2]
    df_clinical_istudy=pd.read_csv(istudypath,sep='\t',comment='#',dtype=str)
    # Remove longitudinal samples
    N_LongiSamples[Study_ID]=(len(df_clinical_istudy.PATIENT_ID)-setlen(df_clinical_istudy.PATIENT_ID)) # Record how many longitudinal samples are removed within every study
    df_clinical_istudy=df_clinical_istudy.drop_duplicates(subset='PATIENT_ID',keep='first')
    df_clinical_istudy['STUDY_ID']=Study_ID
    keepcols=['STUDY_ID','PATIENT_ID','SAMPLE_ID','CODE'] # These columns are ALWAYS present in the curated clinical file.
    list_df_clinical=list_df_clinical+[df_clinical_istudy[keepcols]]
    del df_clinical_istudy
df_clinical=pd.concat(list_df_clinical).reset_index(drop=True) # A unified clinical matrix with ALL the clinical data.
print('Total Number of Studies:',len(list_study_dirs),'\nClinical Data Avaliable for Samples(=Patients or cases)):',len(df_clinical),'\nNumber of Longitudinal Samples Dropped: ',sum(N_LongiSamples.values()))

Total Number of Studies: 139 
Clinical Data Avaliable for Samples(=Patients or cases)): 26429 
Number of Longitudinal Samples Dropped:  988
CPU times: total: 266 ms
Wall time: 360 ms


In [6]:
# Note list of histological codes
histcodes=list(set(df_clinical.CODE.sort_values()))

In [7]:
# Import cbioportal studies and sort reverse chronologically
df_cbiostudies=pd.read_excel(os.path.join(cwd_refdata,'Table2_v6.xlsx'),dtype=str)
df_cbiostudies['Year']=df_cbiostudies['Year'].astype(int)

In [8]:
df_clinical['Year']=[df_cbiostudies.Year[df_cbiostudies.Study_ID==stid].values[0] for stid in df_clinical.STUDY_ID.values]

In [9]:
%%time
# Filter repeated samples acrross studies within a given histological code. Note that this filter assumes that dataframes preseve indices upon copying and slicing. If that functinality changes, the codes need to change.
list_redunPID=[]
df_clinical_NR=df_clinical[:].copy() # initialize non redundant clinical samples
for icode in histcodes:
    # Find all patient IDs within a given hist code
    df_patient=df_clinical[df_clinical.CODE==icode]
    # Find repeating patient IDs
    for pid in set(df_patient.PATIENT_ID.values):
        if sum(df_patient.PATIENT_ID==pid)>1:
            df_tmp_remove=df_patient[df_patient.PATIENT_ID==pid].sort_values('Year',ascending=False)
            for idx_drop in df_tmp_remove.index[1:]:
                df_clinical_NR.drop(idx_drop,inplace=True)
            del df_tmp_remove
    df_patient=df_patient[[sum(df_patient.PATIENT_ID==pid)>1 for pid in df_patient.PATIENT_ID.values]].sort_values('SAMPLE_ID')
    list_redunPID=list_redunPID+[df_patient]    
    del df_patient

CPU times: total: 9.31 s
Wall time: 9.31 s


In [10]:
print('Clinical Data Processed. Number of Redundant Samples:',sum([len(dfi) for dfi in list_redunPID]))

Clinical Data Processed. Number of Redundant Samples: 6809


In [11]:
df_redundants=pd.concat(list_redunPID).reset_index(drop=True)

In [12]:
print('Set of ROSETTA codes with redundant samples and corresponding studies:\n',[[icode]+list(set(df_redundants.STUDY_ID[df_redundants.CODE==icode])) for icode in histcodes if len(set(df_redundants.STUDY_ID[df_redundants.CODE==icode]))>0])

Set of ROSETTA codes with redundant samples and corresponding studies:
 [['96803', 'dlbcl_dfci_2018', 'dlbc_broad_2012'], ['70347', 'stad_uhongkong', 'stes_tcga_pub', 'stad_pfizer_uhongkong', 'stad_tcga_pan_can_atlas_2018'], ['70327', 'prad_broad_2013', 'prad_p1000', 'prad_su2c_2015', 'prad_cpcg_2017', 'prad_tcga_pan_can_atlas_2018', 'prad_broad'], ['94403', 'lgggbm_tcga_pub', 'gbm_tcga_pan_can_atlas_2018'], ['70217', 'stes_tcga_pub', 'esca_tcga_pan_can_atlas_2018'], ['70237', 'lusc_tcga_pan_can_atlas_2018', 'nsclc_tcga_broad_2016'], ['70387', 'paad_icgc', 'paad_qcmg_uq_2016'], ['70337', 'stes_tcga_pub', 'esca_tcga_pan_can_atlas_2018'], ['70137', 'sclc_clcgp', 'sclc_ucologne_2015'], ['95003', 'nbl_broad_2013', 'nbl_target_2018_pub'], ['70107', 'skcm_broad_dfarber', 'skcm_broad'], ['70357', 'luad_broad', 'luad_tcga_pan_can_atlas_2018', 'nsclc_tcga_broad_2016']]


#### Cross check that the removal of redundant Patient IDs was successful:

In [13]:
print("Number of Patient IDs which have been counted more than once:", len([pid for pid in set(df_redundants.PATIENT_ID) if sum(df_clinical.PATIENT_ID==pid)>1]))
print("Validation: Number of redundant samples in post-filtration :",len([pid for pid in set(df_redundants.PATIENT_ID) if sum(df_clinical_NR.PATIENT_ID==pid)>1]))

Number of Patient IDs which have been counted more than once: 3260
Validation: Number of redundant samples in post-filtration : 0


In [15]:
df_redundants.to_excel(os.path.join(cwd_interim,'Redundant_Patient_ID.xlsx'))

## Part 2: Count Mutations

### Import Data

In [16]:
%%time
# List All genes
listCNA_file=[] # list of CNA data genes vs sample IDs
colabsent=[] # LIST OF CASES not present in clinical data but analyzed within CNA
remlist=['?',' ','UNKNOWN',np.nan,'NA','NAN'] # These gene ids are removed as they are not meaningful
iter1=0
for istudyRaw in list_study_dirs:
    # import the CNA file
    df_CNA_istudy=pd.read_csv(os.path.join(cwd_raw_data,istudyRaw,'data_CNA.txt'),sep='\t',dtype=str,comment='#',encoding='cp1252',quoting=3,keep_default_na=False)

    # limit to clinically curated samples
    df_clinical_istudy=df_clinical_NR[df_clinical_NR.STUDY_ID==istudyRaw]
    colfilter=[]
    for icol in df_CNA_istudy.columns:
        if (icol == 'Hugo_Symbol') or (icol in df_clinical_istudy.SAMPLE_ID.values):
            colfilter=colfilter+[icol]
        elif ('Entrez'not in icol) and ('Cyto' not in icol):
            colabsent=colabsent+[[istudyRaw,icol]]
    df_CNA_istudy=df_CNA_istudy[colfilter]

    # Add ICD-code information to the mutation data
    df_CNA_istudy.loc['CODE']=[(df_clinical_istudy.CODE[df_clinical_istudy.SAMPLE_ID==sid].values[0]) if (sid !='Hugo_Symbol') else 'metadata_code' for sid in df_CNA_istudy.columns]

    # store mutations file in memory
    df_CNA_istudy.loc['STUDY_ID']=istudyRaw
    df_CNA_istudy.loc['STUDY_ID','Hugo_Symbol']='metadata_studyID'
    df_CNA_istudy.Hugo_Symbol=df_CNA_istudy.Hugo_Symbol.astype(str).apply(lambda x:x.upper())

    # Include only Unique genes within each sample
    df_CNA_istudy.drop_duplicates(subset='Hugo_Symbol',keep='first',inplace=True) # drop cases with whole row duplicates  - later on check for non-trivial gene duplicates and remove within a study, if they exist.
    # remove nonsense genes
    df_CNA_istudy=df_CNA_istudy[~(df_CNA_istudy.Hugo_Symbol.isin(remlist))]
    listCNA_file=listCNA_file+[df_CNA_istudy]

    del df_CNA_istudy
    del df_clinical_istudy
    
    iter1+=1
    clear_output()
    print('Study Number ',iter1,' done:', istudyRaw)# track progress in case of errors.

Study Number  56  done: wt_target_2018_pub
Wall time: 2min 39s


In [17]:
len(colabsent)

3249

In [81]:
df_absent=pd.DataFrame(colabsent,columns=['STUDY_ID','SAMPLE_ID'])

In [129]:
# find any samples that are present in CNA but the patient_ID is absent in the non-redundant clinical data. Those are the cases which are truly new, otherwise, they are longitudinal or redundant samples.
subSidlist=[]
for stid,sid in colabsent:
#     PidinSid=any([pid in sid for pid in df_clinical_NR[df_clinical_NR.STUDY_ID==stid].PATIENT_ID.values])
    if sid not in df_clinical.SAMPLE_ID.values:
        subSidlist=subSidlist+[[stid,sid]]

In [128]:
len(subSidlist)

532

In [130]:
[[istudy,sum(df_absent.STUDY_ID==istudy)] for istudy in set(df_absent.STUDY_ID)]

[['prad_mich', 2],
 ['blca_tcga_pub_2017', 4],
 ['brca_mbcproject_wagle_2017', 57],
 ['gbm_tcga_pan_can_atlas_2018', 1],
 ['blca_cornell_2016', 31],
 ['mel_tsam_liang_2017', 20],
 ['brca_tcga_pan_can_atlas_2018', 1],
 ['prad_broad', 103],
 ['paad_tcga_pan_can_atlas_2018', 1],
 ['lgggbm_tcga_pub', 628],
 ['angs_project_painter_2018', 12],
 ['tgct_tcga_pan_can_atlas_2018', 21],
 ['wt_target_2018_pub', 5],
 ['thym_tcga_pan_can_atlas_2018', 18],
 ['coadread_tcga_pan_can_atlas_2018', 3],
 ['prad_fhcrc', 95],
 ['skcm_tcga_pan_can_atlas_2018', 1],
 ['nsclc_tcga_broad_2016', 962],
 ['ucs_tcga_pan_can_atlas_2018', 32],
 ['lusc_tcga_pan_can_atlas_2018', 3],
 ['aml_target_2018_pub', 35],
 ['cesc_tcga_pan_can_atlas_2018', 4],
 ['luad_broad', 143],
 ['prad_broad_2013', 10],
 ['pcpg_tcga_pan_can_atlas_2018', 117],
 ['prad_tcga_pan_can_atlas_2018', 449],
 ['stes_tcga_pub', 378],
 ['sarc_tcga_pan_can_atlas_2018', 2],
 ['prad_su2c_2015', 111]]

In [131]:
df_absent1=pd.DataFrame(subSidlist,columns=['STUDY_ID','SAMPLE_ID'])

In [132]:
[[istudy,sum(df_absent1.STUDY_ID==istudy)] for istudy in set(df_absent1.STUDY_ID)]

[['prad_mich', 2],
 ['blca_tcga_pub_2017', 4],
 ['brca_mbcproject_wagle_2017', 57],
 ['gbm_tcga_pan_can_atlas_2018', 1],
 ['blca_cornell_2016', 31],
 ['mel_tsam_liang_2017', 20],
 ['brca_tcga_pan_can_atlas_2018', 1],
 ['paad_tcga_pan_can_atlas_2018', 1],
 ['lgggbm_tcga_pub', 66],
 ['angs_project_painter_2018', 12],
 ['tgct_tcga_pan_can_atlas_2018', 21],
 ['wt_target_2018_pub', 5],
 ['thym_tcga_pan_can_atlas_2018', 18],
 ['coadread_tcga_pan_can_atlas_2018', 3],
 ['prad_fhcrc', 80],
 ['skcm_tcga_pan_can_atlas_2018', 1],
 ['ucs_tcga_pan_can_atlas_2018', 32],
 ['lusc_tcga_pan_can_atlas_2018', 3],
 ['aml_target_2018_pub', 35],
 ['cesc_tcga_pan_can_atlas_2018', 4],
 ['luad_broad', 16],
 ['pcpg_tcga_pan_can_atlas_2018', 117],
 ['sarc_tcga_pan_can_atlas_2018', 2]]

In [106]:
sum([len(idf.columns)-1 for idf in listCNA_file])

12827

In [19]:
'Redundant gene entries exist for these studies:',[idf.Hugo_Symbol.metadata_study_ID[0] for idf in listCNA_file if setlen(idf.index)!=len(idf.index) ]

('Redundant gene entries exist for these studies:', [])

In [20]:
listCNA1=[idf.set_index('Hugo_Symbol',drop=True) for idf in listCNA_file]

In [21]:
# List All genes
geneset=list(set([igene for dfi in listCNA_file for igene in dfi.Hugo_Symbol if 'METADATA' not in igene]))
geneset=sorted(geneset)
print('Number of genes included in our study: ',len(geneset)) 

Number of genes included in our study:  50931


### Perform Counts

In [22]:
from multiprocess import Pool, cpu_count

In [23]:
multi_hist_studylist=[dfi.loc['METADATA_STUDYID'][0] for dfi in listCNA1 if setlen(dfi.loc['METADATA_CODE'])>1]

In [24]:
%%time
# define null dataframes for each category
foutnames={'-2':'homdel','-1':'hemidel','0':'neutral','1':'gain','2':'highAmp'}
dic_dfOut={idf_name:pd.DataFrame(0,columns=['All']+histcodes,index=geneset+['Total'],dtype=int) for idf_name in foutnames.keys()}

# Serial implementation
for dfi in listCNA1[:1]:
    igene_list=[igene for igene in dfi.index if 'METADATA' not in igene]
    icode_list=list(set([ihist for ihist in dfi.loc['METADATA_CODE']]))
    for icode in icode_list:
        for igene in igene_list:
            dfi_icode=dfi.iloc[:,(dfi.loc['METADATA_CODE']==icode).values]
            for idic in dic_dfOut.keys():
                ncases=sum(dfi_icode.loc[igene]==idic)
                dic_dfOut[idic].loc[igene,icode]+=ncases # add the number of samples which have the CNA value to the respective gene, hist location in the respective CNA matrix
                dic_dfOut[idic].loc[igene,'All']+=ncases # add the number of samples which have the CNA value to the respective gene, All location in the respective CNA matrix to keep track of sum over all histologies for each gene

Wall time: 16.6 s


In [25]:
%%time
# define null dataframes for each category
foutnames={'-2':'homdel','-1':'hemidel','0':'neutral','1':'gain','2':'highAmp'}
dic_dfOut={idf_name:pd.DataFrame(0,columns=['All']+histcodes,index=geneset+['Total'],dtype=int) for idf_name in foutnames.keys()}

# Parallel implementation
def process_df(inpvec):
    dic_dfOut=inpvec[0]
    dfi=inpvec[1]
    igene_list=[igene for igene in dfi.index if 'METADATA' not in igene]
    icode_list=list(set([ihist for ihist in dfi.loc['METADATA_CODE']]))
    for icode in icode_list:
        for igene in igene_list:
            dfi_icode=dfi.iloc[:,(dfi.loc['METADATA_CODE']==icode).values]
            for idic in dic_dfOut.keys():
                ncases=sum(dfi_icode.loc[igene]==idic)
                dic_dfOut[idic].loc[igene,icode]+=ncases # add the number of samples which have the CNA value to the respective gene, hist location in the respective CNA matrix
                dic_dfOut[idic].loc[igene,'All']+=ncases # add the number of samples which have the CNA value to the respective gene, All location in the respective CNA matrix to keep track of sum over all histologies for each gene
    return dic_dfOut

ncores=cpu_count()-2# number of cores the task can be split into
datalist=[[cp.deepcopy(dic_dfOut),idf] for idf in listCNA1]
if __name__ == '__main__': #I don't quite understand why this is necessary but it is a part of multiprocessing docs
    po=Pool(ncores) # invoke 5 pooled threads/processes. 
    list_resdicdf=list(po.map(process_df,datalist)) 
    po.close() 
    po.join()

Wall time: 1d 23h 6min 24s


##### It takes 2 days for six cores parallely processing to go through 56 files and find every instance of -2 to 2? That doesn't sound right. Above code needs improvement. 

In [28]:
len(list_resdicdf)

56

In [31]:
with open(os.path.join(cwd_interim,'list_resdicdf.txt'),'w') as fout:
    fout.writelines([str(idic) for idic in list_resdicdf])

In [34]:
dic_dfOut1={idf_name:pd.DataFrame(0,columns=['All']+histcodes,index=geneset+['Total'],dtype=int) for idf_name in foutnames.keys()}
for idic in list_resdicdf:
    for ikey in dic_dfOut1.keys():
        dic_dfOut1[ikey]=dic_dfOut1[ikey]+idic[ikey]

In [35]:
#Check correct counts: All these numbers are same if all mutations were counted once
sum([df_Out.drop(columns='All').sum().sum() for df_Out in dic_dfOut1.values()]),sum([df_Out['All'].sum() for df_Out in dic_dfOut1.values()]),sum([(len(dfi.index)-2)*(len(dfi.columns)-1) for dfi in listCNA_file])

(303894433, 303894433, 304008415)

In [69]:
%%time
# Count number of cases 'Total' within each histology : Perform this action after filtering for curated and sequenced samples
histvec=[elem for idf in listCNA1 for elem in idf.loc['METADATA_CODE'].values]
TotalRow=[sum([len(idf.columns)-1 for idf in listCNA_file])]+[sum([elem==ihist for elem in histvec]) for ihist in histcodes]
for ikey in dic_dfOut1.keys():
    dic_dfOut1[ikey].loc['Total']=TotalRow

Wall time: 135 ms


In [78]:
print('Check that the total counted by counting the number of patients is the same as that counted by summing histogram of included histologies.',
      '\nThis number is 2 to make that happen:',sum(TotalRow)/TotalRow[0])

Check that the total counted by counting the number of patients is the same as that counted by summing histogram of included histologies. 
This number is 2 to make that happen: 2.0


In [72]:
for ikey in dic_dfOut1.keys():
    dic_dfOut1[ikey].to_csv(os.path.join(cwd0,'Genomics_Output_Processed_'+ikey+'.csv'),index_label='Hugo_Symbol')